# 📊 Analyse des Résultats - EOD Zootopia

Ce notebook présente une analyse complète des résultats d'optimisation du dispatching électrique pour le parc Zootopia.

**Contenu :**
- 📈 Statistiques annuelles par scénario
- 🔋 Mix énergétique et production par filière
- 💧 Gestion hydraulique (lacs, fil de l'eau, STEP)
- 🎯 Analyse des tranches d'eau (socle/surplus)
- ⚠️ Défaillances et contraintes
- 📅 Analyses temporelles (journalières, mensuelles)
- 🔄 Comparaison des scénarios (dry/normal/wet)

## 📚 Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Taille des figures
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✅ Bibliothèques chargées avec succès")

## 📂 Chargement des Données

In [ ]:
# Charger les résultats des 3 scénarios
scenarios = ['dry', 'normal', 'wet']
data = {}

for scenario in scenarios:
    try:
        df = pd.read_csv(f'./results/results_{scenario}.csv')
        data[scenario] = df
        print(f"✅ Scénario {scenario:8s} : {len(df):5d} heures chargées")
    except FileNotFoundError:
        print(f"❌ Fichier results_{scenario}.csv introuvable")

# Vérifier les colonnes disponibles
if len(data) > 0:
    print(f"\n📋 Colonnes disponibles ({len(data[scenarios[0]].columns)}) :")
    print(list(data[scenarios[0]].columns))

## 📊 Section 1 : Statistiques Annuelles Globales

In [ ]:
# Calculer les statistiques annuelles pour chaque scénario
stats_annuelles = []

for scenario, df in data.items():
    stats = {
        'Scénario': scenario.upper(),
        'Demande (TWh)': df['load'].sum() / 1e6,
        'Nucléaire (TWh)': df['P_nuc'].sum() / 1e6,
        'Charbon (TWh)': df['P_charbon'].sum() / 1e6,
        'Gaz CCG (TWh)': df['P_ccg'].sum() / 1e6,
        'Gaz TAC (TWh)': df['P_tac'].sum() / 1e6,
        'Fioul (TWh)': df['P_fioul'].sum() / 1e6,
        'Éolien (TWh)': df['P_eolien'].sum() / 1e6,
        'Solaire (TWh)': df['P_solaire'].sum() / 1e6,
        'Hydro FDE (TWh)': df['Phy_fdl'].sum() / 1e6,
        '🆕 Hydro Lacs (TWh)': df['Phy_lac'].sum() / 1e6,
        'STEP décharge (TWh)': df['Pdecharge_STEP'].sum() / 1e6,
        'STEP charge (TWh)': df['Pcharge_STEP'].sum() / 1e6,
        'Défaillance (MWh)': df['Puns'].sum(),
        'Spill (TWh)': df['Pspill'].sum() / 1e6,
    }
    stats_annuelles.append(stats)

df_stats = pd.DataFrame(stats_annuelles).set_index('Scénario')

print("="*100)
print("📊 STATISTIQUES ANNUELLES PAR SCÉNARIO")
print("="*100)
print(df_stats.round(2).to_string())
print("\n")

## 🎨 Section 2 : Mix Énergétique Annuel

In [ ]:
# Mix énergétique pour chaque scénario
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (scenario, df) in enumerate(data.items()):
    ax = axes[idx]
    
    # Calculer les productions totales (TWh)
    productions = {
        'Nucléaire': df['P_nuc'].sum() / 1e6,
        'Charbon': df['P_charbon'].sum() / 1e6,
        'Gaz CCG': df['P_ccg'].sum() / 1e6,
        'Fioul': df['P_fioul'].sum() / 1e6,
        'Éolien': df['P_eolien'].sum() / 1e6,
        'Solaire': df['P_solaire'].sum() / 1e6,
        'Hydro FDE': df['Phy_fdl'].sum() / 1e6,
        'Hydro Lacs': df['Phy_lac'].sum() / 1e6,
        'STEP': df['Pdecharge_STEP'].sum() / 1e6,
        'Déchets': df['P_dechets'].sum() / 1e6,
        'Biomasse': df['P_biomasse'].sum() / 1e6,
    }
    
    # Filtrer les valeurs significatives
    productions = {k: v for k, v in productions.items() if v > 0.1}
    
    # Pie chart
    colors = plt.cm.Set3(range(len(productions)))
    wedges, texts, autotexts = ax.pie(
        productions.values(), 
        labels=productions.keys(),
        autopct='%1.1f%%',
        startangle=90,
        colors=colors
    )
    
    # Améliorer la lisibilité
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(9)
        autotext.set_weight('bold')
    
    ax.set_title(f'Scénario {scenario.upper()}\n({sum(productions.values()):.1f} TWh)', 
                 fontsize=12, fontweight='bold')

plt.suptitle('🎨 Mix Énergétique Annuel par Scénario', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('./results/mix_energetique.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Graphique sauvegardé : ./results/mix_energetique.png")

## 📈 Section 3 : Production Horaire par Filière (Semaine Type)

In [ ]:
# Sélectionner une semaine représentative (ex: heure 2000-2168)
week_start = 2000
week_end = week_start + 168

fig, axes = plt.subplots(3, 1, figsize=(16, 12))

for idx, (scenario, df) in enumerate(data.items()):
    ax = axes[idx]
    
    # Extraire la semaine
    df_week = df[(df['hour_global'] >= week_start) & (df['hour_global'] < week_end)].copy()
    df_week = df_week.reset_index(drop=True)
    
    # Empiler les productions
    x = df_week.index
    
    ax.fill_between(x, 0, df_week['P_nuc'], label='Nucléaire', alpha=0.8)
    
    y_bottom = df_week['P_nuc']
    ax.fill_between(x, y_bottom, y_bottom + df_week['P_charbon'], 
                     label='Charbon', alpha=0.8)
    
    y_bottom += df_week['P_charbon']
    ax.fill_between(x, y_bottom, y_bottom + df_week['P_ccg'], 
                     label='Gaz CCG', alpha=0.8)
    
    y_bottom += df_week['P_ccg']
    ax.fill_between(x, y_bottom, y_bottom + df_week['Phy_fdl'], 
                     label='Hydro FDE', alpha=0.8)
    
    y_bottom += df_week['Phy_fdl']
    ax.fill_between(x, y_bottom, y_bottom + df_week['Phy_lac'], 
                     label='🆕 Hydro Lacs', alpha=0.8, color='dodgerblue')
    
    y_bottom += df_week['Phy_lac']
    ax.fill_between(x, y_bottom, y_bottom + df_week['P_eolien'], 
                     label='Éolien', alpha=0.8)
    
    y_bottom += df_week['P_eolien']
    ax.fill_between(x, y_bottom, y_bottom + df_week['P_solaire'], 
                     label='Solaire', alpha=0.8)
    
    # Tracer la demande
    ax.plot(x, df_week['load'], 'r--', linewidth=2, label='Demande', alpha=0.9)
    
    ax.set_title(f'Scénario {scenario.upper()} - Semaine heures {week_start}-{week_end}', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Heure de la semaine')
    ax.set_ylabel('Puissance (MW)')
    ax.legend(loc='upper left', ncol=4, fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Marquer les jours
    for day in range(0, 168, 24):
        ax.axvline(day, color='gray', linestyle=':', alpha=0.5, linewidth=0.8)

plt.suptitle('📈 Production Horaire par Filière - Semaine Type', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('./results/production_semaine_type.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Graphique sauvegardé : ./results/production_semaine_type.png")

## 💧 Section 4 : Gestion Hydraulique - Stocks et Tranches d'Eau

In [ ]:
# Analyse des stocks hydrauliques et tranches
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

for idx, (scenario, df) in enumerate(data.items()):
    # Graphique 1 : Stock hydro total avec min/max
    ax1 = axes[idx, 0]
    x = df['hour_global']
    
    ax1.fill_between(x, df['stock_hydro_min']/1e6, df['stock_hydro_max']/1e6, 
                      alpha=0.2, color='lightblue', label='Plage autorisée')
    ax1.plot(x, df['stock_hydro']/1e6, 'b-', linewidth=1.5, label='Stock réel', alpha=0.8)
    
    ax1.set_title(f'{scenario.upper()} - Stock Hydraulique', fontweight='bold')
    ax1.set_xlabel('Heure annuelle')
    ax1.set_ylabel('Stock (TWh)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Graphique 2 : Tranches d'eau (Socle vs Surplus)
    ax2 = axes[idx, 1]
    
    ax2.fill_between(x, 0, df['S_socle']/1e6, 
                      alpha=0.7, color='navy', label='🆕 Tranche Socle')
    ax2.fill_between(x, df['S_socle']/1e6, df['stock_hydro']/1e6, 
                      alpha=0.7, color='cyan', label='🆕 Tranche Surplus')
    
    ax2.set_title(f'{scenario.upper()} - Tranches d\'Eau', fontweight='bold')
    ax2.set_xlabel('Heure annuelle')
    ax2.set_ylabel('Stock (TWh)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

plt.suptitle('💧 Gestion Hydraulique - Stocks et Tranches d\'Eau', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('./results/stocks_hydrauliques.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Graphique sauvegardé : ./results/stocks_hydrauliques.png")

## 🔋 Section 5 : STEP - Charge/Décharge et Stock

In [ ]:
# Analyse STEP
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

for idx, (scenario, df) in enumerate(data.items()):
    x = df['hour_global']
    
    # Graphique 1 : Stock STEP
    ax1 = axes[idx, 0]
    ax1.plot(x, df['stock_STEP']/1e6, 'g-', linewidth=1.5, label='Stock STEP')
    ax1.axhline(0.5, color='orange', linestyle='--', alpha=0.7, label='Capacité max (0.5 TWh)')
    ax1.set_title(f'{scenario.upper()} - Stock STEP', fontweight='bold')
    ax1.set_xlabel('Heure annuelle')
    ax1.set_ylabel('Stock (TWh)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Graphique 2 : Charge vs Décharge
    ax2 = axes[idx, 1]
    ax2.fill_between(x, 0, df['Pdecharge_STEP'], 
                      alpha=0.6, color='green', label='Décharge (turbinage)')
    ax2.fill_between(x, 0, -df['Pcharge_STEP'], 
                      alpha=0.6, color='red', label='Charge (pompage)')
    ax2.set_title(f'{scenario.upper()} - STEP Charge/Décharge', fontweight='bold')
    ax2.set_xlabel('Heure annuelle')
    ax2.set_ylabel('Puissance (MW)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.axhline(0, color='black', linewidth=0.8)

plt.suptitle('🔋 Station de Transfert d\'Énergie par Pompage (STEP)', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('./results/step_analyse.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Graphique sauvegardé : ./results/step_analyse.png")

## 🎯 Section 6 : Production Hydraulique des Lacs (Analyse Détaillée)

In [ ]:
# Analyse spécifique de la production hydraulique des lacs
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Graphique 1 : Production horaire des lacs par scénario
ax1 = axes[0, 0]
for scenario, df in data.items():
    ax1.plot(df['hour_global'], df['Phy_lac'], label=f'{scenario.upper()}', alpha=0.7, linewidth=1)
ax1.set_title('🆕 Production Hydraulique des Lacs - Vue Annuelle', fontweight='bold')
ax1.set_xlabel('Heure annuelle')
ax1.set_ylabel('Puissance (MW)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Graphique 2 : Distribution de la production
ax2 = axes[0, 1]
prod_data = [df['Phy_lac'].values for df in data.values()]
ax2.boxplot(prod_data, labels=[s.upper() for s in scenarios])
ax2.set_title('🆕 Distribution Production Lacs', fontweight='bold')
ax2.set_ylabel('Puissance (MW)')
ax2.grid(True, alpha=0.3, axis='y')

# Graphique 3 : Production mensuelle moyenne
ax3 = axes[1, 0]
for scenario, df in data.items():
    monthly_prod = df.groupby('month')['Phy_lac'].mean()
    ax3.plot(monthly_prod.index, monthly_prod.values, marker='o', label=scenario.upper(), linewidth=2)
ax3.set_title('🆕 Production Moyenne Mensuelle des Lacs', fontweight='bold')
ax3.set_xlabel('Mois')
ax3.set_ylabel('Puissance moyenne (MW)')
ax3.set_xticks(range(1, 13))
ax3.legend()
ax3.grid(True, alpha=0.3)

# Graphique 4 : Production totale vs apports
ax4 = axes[1, 1]
x_pos = np.arange(len(scenarios))
prod_totals = [df['Phy_lac'].sum()/1e6 for df in data.values()]
inflow_totals = [df['inflows_lac'].sum()/1e6 for df in data.values()]

width = 0.35
ax4.bar(x_pos - width/2, prod_totals, width, label='Production', color='dodgerblue')
ax4.bar(x_pos + width/2, inflow_totals, width, label='Apports', color='lightgreen')
ax4.set_title('🆕 Production vs Apports Annuels', fontweight='bold')
ax4.set_ylabel('Énergie (TWh)')
ax4.set_xticks(x_pos)
ax4.set_xticklabels([s.upper() for s in scenarios])
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

plt.suptitle('🎯 Analyse Détaillée - Production Hydraulique des Lacs', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('./results/production_lacs_analyse.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Graphique sauvegardé : ./results/production_lacs_analyse.png")

# Statistiques détaillées
print("\n📊 STATISTIQUES PRODUCTION HYDRAULIQUE LACS")
print("="*70)
for scenario, df in data.items():
    print(f"\nScénario {scenario.upper()}:")
    print(f"  Production totale    : {df['Phy_lac'].sum()/1e6:.2f} TWh")
    print(f"  Production moyenne   : {df['Phy_lac'].mean():.1f} MW")
    print(f"  Production max       : {df['Phy_lac'].max():.1f} MW")
    print(f"  Heures de production : {(df['Phy_lac'] > 0).sum()} h ({(df['Phy_lac'] > 0).sum()/len(df)*100:.1f}%)")
    print(f"  Taux utilisation     : {df['Phy_lac'].mean()/6000*100:.1f}% (Pmax=6000 MW)")

## ⚠️ Section 7 : Défaillances et Violations de Contraintes

In [ ]:
# Analyse des défaillances et slacks
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Graphique 1 : Défaillances par scénario
ax1 = axes[0, 0]
defaillances = [df['Puns'].sum()/1e3 for df in data.values()]
colors = ['red' if d > 0 else 'green' for d in defaillances]
ax1.bar(scenarios, defaillances, color=colors, alpha=0.7)
ax1.set_title('⚠️ Défaillances Totales', fontweight='bold')
ax1.set_ylabel('Énergie non servie (GWh)')
ax1.grid(True, alpha=0.3, axis='y')

# Graphique 2 : Spill (déversement hydraulique)
ax2 = axes[0, 1]
spills = [df['Pspill'].sum()/1e6 for df in data.values()]
ax2.bar(scenarios, spills, color='orange', alpha=0.7)
ax2.set_title('💧 Déversements Hydrauliques', fontweight='bold')
ax2.set_ylabel('Énergie déversée (TWh)')
ax2.grid(True, alpha=0.3, axis='y')

# Graphique 3 : Slacks saisonniers
ax3 = axes[1, 0]
for scenario, df in data.items():
    slack_seasonal_nz = df[df['slack_seasonal'] > 0.1]
    if len(slack_seasonal_nz) > 0:
        ax3.scatter(slack_seasonal_nz['hour_global'], 
                   slack_seasonal_nz['slack_seasonal']/1e6,
                   label=scenario.upper(), alpha=0.6, s=20)
ax3.set_title('🎯 Violations Cible Saisonnière', fontweight='bold')
ax3.set_xlabel('Heure annuelle')
ax3.set_ylabel('Slack (TWh)')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Graphique 4 : Heures infaisables
ax4 = axes[1, 1]
if 'infeasible_flag' in data[scenarios[0]].columns:
    infeasible_counts = [df['infeasible_flag'].sum() for df in data.values()]
    ax4.bar(scenarios, infeasible_counts, color='darkred', alpha=0.7)
    ax4.set_title('🚫 Heures Infaisables', fontweight='bold')
    ax4.set_ylabel('Nombre d\'heures')
    ax4.grid(True, alpha=0.3, axis='y')
else:
    ax4.text(0.5, 0.5, 'Pas de données d\'infaisabilité', 
             ha='center', va='center', transform=ax4.transAxes)
    ax4.axis('off')

plt.suptitle('⚠️ Défaillances et Violations de Contraintes', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('./results/defaillances_contraintes.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Graphique sauvegardé : ./results/defaillances_contraintes.png")

# Résumé textuel
print("\n⚠️ RÉSUMÉ DÉFAILLANCES")
print("="*70)
for scenario, df in data.items():
    total_uns = df['Puns'].sum()
    nb_heures_uns = (df['Puns'] > 0).sum()
    print(f"\n{scenario.upper()}:")
    print(f"  Total défaillance : {total_uns/1e3:.2f} GWh")
    print(f"  Heures défaillantes : {nb_heures_uns} h")
    print(f"  Taux de défaillance : {total_uns/df['load'].sum()*100:.4f}%")

## 📅 Section 8 : Analyses Temporelles - Profils Journaliers et Mensuels

In [ ]:
# Profil journalier moyen
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for idx, (scenario, df) in enumerate(data.items()):
    if idx >= 3:
        break
    
    ax = axes[idx // 2, idx % 2]
    
    # Calculer l'heure de la journée (0-23)
    df['hour_of_day'] = (df['hour_global'] - 1) % 24
    
    # Profils moyens par heure
    hourly_avg = df.groupby('hour_of_day').agg({
        'load': 'mean',
        'P_nuc': 'mean',
        'P_eolien': 'mean',
        'P_solaire': 'mean',
        'Phy_lac': 'mean',
        'Pdecharge_STEP': 'mean'
    })
    
    hours = hourly_avg.index
    ax.plot(hours, hourly_avg['load'], 'r-', linewidth=2.5, label='Demande', marker='o')
    ax.plot(hours, hourly_avg['P_nuc'], 'b--', linewidth=2, label='Nucléaire', marker='s')
    ax.plot(hours, hourly_avg['P_eolien'], 'g--', linewidth=2, label='Éolien', marker='^')
    ax.plot(hours, hourly_avg['P_solaire'], 'orange', linewidth=2, label='Solaire', marker='v')
    ax.plot(hours, hourly_avg['Phy_lac'], 'cyan', linewidth=2, label='🆕 Lacs', marker='d')
    ax.plot(hours, hourly_avg['Pdecharge_STEP'], 'purple', linewidth=2, label='STEP', marker='*')
    
    ax.set_title(f'{scenario.upper()} - Profil Journalier Moyen', fontweight='bold')
    ax.set_xlabel('Heure du jour')
    ax.set_ylabel('Puissance moyenne (MW)')
    ax.set_xticks(range(0, 24, 2))
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)

# Graphique 4 : Comparaison des profils de demande
ax4 = axes[1, 1]
for scenario, df in data.items():
    df['hour_of_day'] = (df['hour_global'] - 1) % 24
    hourly_load = df.groupby('hour_of_day')['load'].mean()
    ax4.plot(hourly_load.index, hourly_load.values, 
             linewidth=2, marker='o', label=scenario.upper())

ax4.set_title('Comparaison Profils de Demande', fontweight='bold')
ax4.set_xlabel('Heure du jour')
ax4.set_ylabel('Puissance moyenne (MW)')
ax4.set_xticks(range(0, 24, 2))
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.suptitle('📅 Profils Journaliers Moyens', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('./results/profils_journaliers.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Graphique sauvegardé : ./results/profils_journaliers.png")

## 🔄 Section 9 : Comparaison des Scénarios - Métriques Clés

In [ ]:
# Tableau comparatif détaillé
comparison = []

for scenario, df in data.items():
    metrics = {
        'Scénario': scenario.upper(),
        'Demande (TWh)': df['load'].sum() / 1e6,
        'Prod. Totale (TWh)': (df['P_nuc'].sum() + df['P_charbon'].sum() + 
                                df['P_ccg'].sum() + df['P_tac'].sum() +
                                df['P_fioul'].sum() + df['P_eolien'].sum() + 
                                df['P_solaire'].sum() + df['Phy_fdl'].sum() +
                                df['Phy_lac'].sum() + df['Pdecharge_STEP'].sum() +
                                df['P_dechets'].sum() + df['P_biomasse'].sum()) / 1e6,
        '% Nucléaire': df['P_nuc'].sum() / df['load'].sum() * 100,
        '% ENR': (df['P_eolien'].sum() + df['P_solaire'].sum() + 
                  df['Phy_fdl'].sum() + df['Phy_lac'].sum()) / df['load'].sum() * 100,
        '% Fossiles': (df['P_charbon'].sum() + df['P_ccg'].sum() + 
                       df['P_tac'].sum() + df['P_fioul'].sum()) / df['load'].sum() * 100,
        'Facteur charge Nuc (%)': df['P_nuc'].mean() / 6200 * 100,  # 6200 MW capacité totale
        'Facteur charge Éol (%)': df['P_eolien'].mean() / 6400 * 100,
        'Facteur charge Sol (%)': df['P_solaire'].mean() / 3000 * 100,
        'Stock Hydro min (TWh)': df['stock_hydro'].min() / 1e6,
        'Stock Hydro max (TWh)': df['stock_hydro'].max() / 1e6,
        'Stock Hydro moy (TWh)': df['stock_hydro'].mean() / 1e6,
        'STEP cycles': df['Pcharge_STEP'].sum() / 1e6 / 0.5,  # Nombre de cycles complets
        'Défaillance (MWh)': df['Puns'].sum(),
    }
    comparison.append(metrics)

df_comp = pd.DataFrame(comparison).set_index('Scénario')

print("="*100)
print("🔄 COMPARAISON DÉTAILLÉE DES SCÉNARIOS")
print("="*100)
print(df_comp.round(2).to_string())
print("\n")

# Graphique radar pour comparaison visuelle
from math import pi

categories = ['% Nucléaire', '% ENR', 'FC Nuc', 'FC Éol', 'Stock Hydro']
N = len(categories)

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

for scenario, df in data.items():
    values = [
        df['P_nuc'].sum() / df['load'].sum() * 100,
        (df['P_eolien'].sum() + df['P_solaire'].sum() + 
         df['Phy_fdl'].sum() + df['Phy_lac'].sum()) / df['load'].sum() * 100,
        df['P_nuc'].mean() / 6200 * 100,
        df['P_eolien'].mean() / 6400 * 100,
        df['stock_hydro'].mean() / 1e6 / 1.0 * 100  # % de la capacité
    ]
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2, label=scenario.upper())
    ax.fill(angles, values, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=10)
ax.set_ylim(0, 100)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.set_title('🔄 Comparaison Radar des Scénarios', fontsize=14, fontweight='bold', pad=20)
ax.grid(True)

plt.tight_layout()
plt.savefig('./results/comparaison_scenarios_radar.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Graphique sauvegardé : ./results/comparaison_scenarios_radar.png")

## 💰 Section 10 : Analyse Économique (Coûts de Production)

In [ ]:
# Coûts marginaux par technologie
prix_techno = {
    'Nucléaire': 12,
    'Charbon': 36,
    'Gaz CCG': 40,
    'Gaz TAC': 70,
    'Cogénération': 70,
    'Fioul': 100,
    'Éolien': 0,
    'Solaire': 0,
    'Hydro': 0,
    'Déchets': 0,
    'Biomasse': 0
}

# Calculer les coûts par scénario
couts_scenarios = []

for scenario, df in data.items():
    cout_nuc = df['P_nuc'].sum() * prix_techno['Nucléaire']
    cout_charbon = df['P_charbon'].sum() * prix_techno['Charbon']
    cout_gaz = df['P_ccg'].sum() * prix_techno['Gaz CCG'] + df['P_tac'].sum() * prix_techno['Gaz TAC']
    cout_fioul = df['P_fioul'].sum() * prix_techno['Fioul']
    cout_cogen = df['P_cogen'].sum() * prix_techno['Cogénération']
    
    cout_total = cout_nuc + cout_charbon + cout_gaz + cout_fioul + cout_cogen
    
    couts_scenarios.append({
        'Scénario': scenario.upper(),
        'Coût Nucléaire (M€)': cout_nuc / 1e6,
        'Coût Charbon (M€)': cout_charbon / 1e6,
        'Coût Gaz (M€)': cout_gaz / 1e6,
        'Coût Fioul (M€)': cout_fioul / 1e6,
        'Coût Cogén (M€)': cout_cogen / 1e6,
        'Coût Total (M€)': cout_total / 1e6,
        'Coût moyen (€/MWh)': cout_total / df['load'].sum()
    })

df_couts = pd.DataFrame(couts_scenarios).set_index('Scénario')

print("="*100)
print("💰 ANALYSE ÉCONOMIQUE - COÛTS DE PRODUCTION")
print("="*100)
print(df_couts.round(2).to_string())
print("\n")

# Graphique : Décomposition des coûts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Graphique 1 : Coûts totaux par scénario
ax1 = axes[0]
scenarios_list = [s.upper() for s in scenarios]
couts_totaux = df_couts['Coût Total (M€)'].values
colors = plt.cm.viridis(np.linspace(0, 1, len(scenarios)))
ax1.bar(scenarios_list, couts_totaux, color=colors, alpha=0.8)
ax1.set_title('💰 Coûts Totaux par Scénario', fontweight='bold', fontsize=12)
ax1.set_ylabel('Coût (M€)')
ax1.grid(True, alpha=0.3, axis='y')

# Ajouter les valeurs sur les barres
for i, v in enumerate(couts_totaux):
    ax1.text(i, v + max(couts_totaux)*0.02, f'{v:.1f} M€', 
             ha='center', fontweight='bold')

# Graphique 2 : Décomposition par technologie (stacked bar)
ax2 = axes[1]
x_pos = np.arange(len(scenarios))
width = 0.6

bottom = np.zeros(len(scenarios))
techno_colors = {
    'Coût Nucléaire (M€)': 'blue',
    'Coût Charbon (M€)': 'black',
    'Coût Gaz (M€)': 'orange',
    'Coût Fioul (M€)': 'red',
    'Coût Cogén (M€)': 'yellow'
}

for col, color in techno_colors.items():
    values = df_couts[col].values
    ax2.bar(scenarios_list, values, width, bottom=bottom, 
            label=col.replace(' (M€)', ''), color=color, alpha=0.8)
    bottom += values

ax2.set_title('💰 Décomposition des Coûts par Technologie', fontweight='bold', fontsize=12)
ax2.set_ylabel('Coût (M€)')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('💰 Analyse Économique des Scénarios', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('./results/analyse_economique.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Graphique sauvegardé : ./results/analyse_economique.png")

## 🌡️ Section 11 : Analyse de Sensibilité - Impact des Apports Hydrauliques

In [ ]:
# Analyse de l'impact des apports sur les différentes métriques
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Extraire les facteurs d'apport par scénario
apport_factors = {'dry': 0.5, 'normal': 0.7, 'wet': 1.0}
scenarios_ordered = ['dry', 'normal', 'wet']
x_factors = [apport_factors[s] for s in scenarios_ordered]

# Métriques à analyser
metrics = {
    'Prod Lacs (TWh)': [data[s]['Phy_lac'].sum()/1e6 for s in scenarios_ordered],
    'Stock moyen (TWh)': [data[s]['stock_hydro'].mean()/1e6 for s in scenarios_ordered],
    'Prod Fossiles (TWh)': [(data[s]['P_charbon'].sum() + data[s]['P_ccg'].sum() + 
                              data[s]['P_fioul'].sum())/1e6 for s in scenarios_ordered],
    'Défaillance (GWh)': [data[s]['Puns'].sum()/1e3 for s in scenarios_ordered]
}

for idx, (metric_name, values) in enumerate(metrics.items()):
    ax = axes[idx // 2, idx % 2]
    ax.plot(x_factors, values, marker='o', linewidth=2, markersize=10, color='darkblue')
    ax.set_title(f'Impact sur: {metric_name}', fontweight='bold')
    ax.set_xlabel('Facteur d\'apport hydraulique')
    ax.set_ylabel(metric_name)
    ax.grid(True, alpha=0.3)
    
    # Annoter les points
    for i, (x, y) in enumerate(zip(x_factors, values)):
        ax.annotate(scenarios_ordered[i].upper(), (x, y), 
                   textcoords="offset points", xytext=(0, 10), 
                   ha='center', fontsize=9, fontweight='bold')

plt.suptitle('🌡️ Analyse de Sensibilité - Impact des Apports Hydrauliques', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('./results/sensibilite_apports.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Graphique sauvegardé : ./results/sensibilite_apports.png")

## 📋 Section 12 : Rapport Final et Export

In [ ]:
# Générer un rapport texte complet
rapport = []
rapport.append("="*100)
rapport.append("📋 RAPPORT FINAL - OPTIMISATION DISPATCHING ÉLECTRIQUE ZOOTOPIA")
rapport.append("="*100)
rapport.append("")
rapport.append(f"Date d'analyse : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
rapport.append(f"Nombre de scénarios analysés : {len(data)}")
rapport.append(f"Période simulée : 8760 heures (1 année complète)")
rapport.append("")

rapport.append("\n" + "="*100)
rapport.append("1. RÉSUMÉ EXÉCUTIF")
rapport.append("="*100)

for scenario, df in data.items():
    rapport.append(f"\n{scenario.upper()}:")
    rapport.append(f"  • Demande totale : {df['load'].sum()/1e6:.2f} TWh")
    rapport.append(f"  • Production nucléaire : {df['P_nuc'].sum()/1e6:.2f} TWh ({df['P_nuc'].sum()/df['load'].sum()*100:.1f}%)")
    rapport.append(f"  • Production ENR : {(df['P_eolien'].sum() + df['P_solaire'].sum() + df['Phy_fdl'].sum() + df['Phy_lac'].sum())/1e6:.2f} TWh")
    rapport.append(f"  • 🆕 Production hydraulique lacs : {df['Phy_lac'].sum()/1e6:.2f} TWh")
    rapport.append(f"  • Défaillance : {df['Puns'].sum()/1e3:.3f} GWh ({df['Puns'].sum()/df['load'].sum()*100:.4f}%)")
    rapport.append(f"  • Coût moyen : {df_couts.loc[scenario.upper(), 'Coût moyen (€/MWh)']:.2f} €/MWh")

rapport.append("\n" + "="*100)
rapport.append("2. POINTS CLÉS D'AMÉLIORATION")
rapport.append("="*100)
rapport.append("")
rapport.append("✅ Améliorations intégrées :")
rapport.append("  • Arguments en ligne de commande pour sélection de scénarios")
rapport.append("  • Fonctions de conversion robustes pour lecture Excel")
rapport.append("  • 🆕 Tranches d'eau (socle/surplus) pour meilleure gestion hydraulique")
rapport.append("  • 🆕 Production hydraulique lacs décommentée et exportée")
rapport.append("  • Export enrichi avec nouvelles colonnes d'analyse")
rapport.append("")

rapport.append("\n" + "="*100)
rapport.append("3. RECOMMANDATIONS")
rapport.append("="*100)
rapport.append("")

# Recommandations basées sur les résultats
for scenario, df in data.items():
    if df['Puns'].sum() > 1000:  # Si défaillance > 1 MWh
        rapport.append(f"⚠️ {scenario.upper()}: Défaillance détectée → Augmenter capacité de production")
    
    if (df['P_charbon'].sum() + df['P_ccg'].sum()) / df['load'].sum() > 0.3:
        rapport.append(f"🔥 {scenario.upper()}: Part fossiles élevée ({(df['P_charbon'].sum() + df['P_ccg'].sum()) / df['load'].sum() * 100:.1f}%) → Développer ENR")
    
    if df['stock_hydro'].min() < 0.2e6:
        rapport.append(f"💧 {scenario.upper()}: Stock hydro critique (min: {df['stock_hydro'].min()/1e6:.2f} TWh) → Revoir gestion")

rapport.append("")
rapport.append("\n" + "="*100)
rapport.append("FIN DU RAPPORT")
rapport.append("="*100)

# Sauvegarder le rapport
with open('./results/rapport_analyse.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(rapport))

# Afficher le rapport
print('\n'.join(rapport))

print("\n✅ Rapport sauvegardé : ./results/rapport_analyse.txt")

## 📊 Section 13 : Export des Statistiques en Excel

In [ ]:
# Créer un fichier Excel avec toutes les statistiques
try:
    with pd.ExcelWriter('./results/statistiques_completes.xlsx', engine='openpyxl') as writer:
        # Feuille 1 : Statistiques annuelles
        df_stats.to_excel(writer, sheet_name='Stats_Annuelles')
        
        # Feuille 2 : Comparaison scénarios
        df_comp.to_excel(writer, sheet_name='Comparaison')
        
        # Feuille 3 : Analyse économique
        df_couts.to_excel(writer, sheet_name='Analyse_Economique')
        
        # Feuilles 4-6 : Données horaires agrégées par scénario
        for scenario, df in data.items():
            # Sélectionner colonnes principales
            df_export = df[[
                'hour_global', 'month', 'load', 'P_nuc', 'P_charbon', 'P_ccg',
                'P_eolien', 'P_solaire', 'Phy_fdl', 'Phy_lac',
                'stock_hydro', 'stock_STEP', 'Puns', 'Pspill'
            ]].copy()
            
            # Agréger par jour (moyenne)
            df_export['day'] = (df_export['hour_global'] - 1) // 24 + 1
            df_daily = df_export.groupby('day').agg({
                'month': 'first',
                'load': 'mean',
                'P_nuc': 'mean',
                'P_charbon': 'mean',
                'P_ccg': 'mean',
                'P_eolien': 'mean',
                'P_solaire': 'mean',
                'Phy_fdl': 'mean',
                'Phy_lac': 'mean',
                'stock_hydro': 'mean',
                'stock_STEP': 'mean',
                'Puns': 'sum',
                'Pspill': 'sum'
            })
            
            df_daily.to_excel(writer, sheet_name=f'{scenario.upper()}_daily')
    
    print("✅ Statistiques exportées : ./results/statistiques_completes.xlsx")
    print("   • 3 feuilles de statistiques")
    print("   • 3 feuilles de données journalières (une par scénario)")
    
except Exception as e:
    print(f"❌ Erreur lors de l'export Excel : {e}")
    print("   → Installer openpyxl : pip install openpyxl")

## ✅ Conclusion et Récapitulatif

In [ ]:
print("\n" + "="*100)
print("✅ ANALYSE TERMINÉE AVEC SUCCÈS")
print("="*100)
print("\n📂 Fichiers générés dans ./results/ :")
print("   📊 mix_energetique.png")
print("   📈 production_semaine_type.png")
print("   💧 stocks_hydrauliques.png")
print("   🔋 step_analyse.png")
print("   🎯 production_lacs_analyse.png")
print("   ⚠️  defaillances_contraintes.png")
print("   📅 profils_journaliers.png")
print("   🔄 comparaison_scenarios_radar.png")
print("   💰 analyse_economique.png")
print("   🌡️  sensibilite_apports.png")
print("   📋 rapport_analyse.txt")
print("   📊 statistiques_completes.xlsx")
print("\n🎉 Toutes les analyses et visualisations ont été générées !")
print("="*100)